# RUKOPYS YOLO + Qwen3-VL Hybrid Submit - Kaggle T4x2

Simple Kaggle submit notebook.

Pipeline:

- Read test images from a full `TEST_DIR` path that already points to the `test` folder.
- Run DocLayout-YOLO from a configurable `.pt` weights path to produce `bbox` + `type` regions.
- Load Qwen3-VL 8B Instruct base model and one LoRA adapter from local Kaggle input paths.
- OCR every text crop with Qwen3-VL.
- Split images across GPU workers for Kaggle T4x2.
- Write a CSV output with columns `image,regions` under `/kaggle/working/<RUN_NAME>`.

No external bbox CSV is required.


In [ ]:
INSTALL_DEPS = True
INSTALL_DOCLAYOUT_YOLO = True
DOCLAYOUT_REPO = "/kaggle/working/DocLayout-YOLO"

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "qwen-vl-utils",
            "huggingface_hub",
            "hf_transfer",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)

if INSTALL_DOCLAYOUT_YOLO:
    import subprocess
    import sys
    from pathlib import Path

    repo_candidates = [
        Path(DOCLAYOUT_REPO),
        Path("/kaggle/input/DocLayout-YOLO"),
        Path("/kaggle/input/doclayout-yolo/DocLayout-YOLO"),
    ]
    repo = next((p for p in repo_candidates if p.exists()), Path(DOCLAYOUT_REPO))
    if not repo.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/opendatalab/DocLayout-YOLO.git", str(repo)])
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    if str(repo).startswith("/kaggle/working"):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)])


In [ ]:
import gc
import json
import logging
import math
import os
import re
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path
from types import ModuleType

import pandas as pd
import torch
from PIL import Image
from torchvision.ops import nms
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


def suppress_transformers_noise():
    message = r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*"
    warnings.filterwarnings("ignore", message=message)
    logging.getLogger("transformers").setLevel(logging.ERROR)
    logging.getLogger("transformers.processing_utils").setLevel(logging.ERROR)
    try:
        from transformers.utils import logging as hf_logging

        hf_logging.set_verbosity_error()
    except Exception:
        pass


suppress_transformers_noise()

for repo_path in ["/kaggle/working/DocLayout-YOLO", "/kaggle/input/DocLayout-YOLO", "/kaggle/input/doclayout-yolo/DocLayout-YOLO"]:
    p = Path(repo_path)
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Some DocLayout-YOLO checkpoints reference this callback module while loading.
if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

from doclayout_yolo import YOLOv10


def format_bytes(num_bytes):
    try:
        value = float(num_bytes)
    except Exception:
        return "unknown"
    units = ["B", "KB", "MB", "GB", "TB"]
    idx = 0
    while value >= 1024 and idx < len(units) - 1:
        value /= 1024.0
        idx += 1
    return f"{value:.1f}{units[idx]}"


def format_rate(num_bytes, seconds):
    if not seconds or seconds <= 0:
        return "--/s"
    return f"{format_bytes(num_bytes / seconds)}/s"


def short_duration(seconds):
    seconds = int(round(max(0, seconds)))
    minutes, secs = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def path_tree_stats(path):
    path = Path(path)
    if path.is_file():
        return 1, path.stat().st_size
    files = 0
    total = 0
    if not path.exists():
        return 0, 0
    for item in path.rglob("*"):
        if item.is_file():
            files += 1
            try:
                total += item.stat().st_size
            except OSError:
                pass
    return files, total


def log_path_stats(label, path):
    path = Path(path)
    files, total = path_tree_stats(path)
    exists = path.exists()
    print(f"{label}: {path} exists={exists} files={files:,} size={format_bytes(total)}", flush=True)
    return files, total


def copytree_with_progress(src_dir, dst_dir, label, log_every_bytes=512 * 1024 * 1024, log_every_seconds=10):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    files_total, bytes_total = path_tree_stats(src_dir)
    print(f"{label}: copy start files={files_total:,} size={format_bytes(bytes_total)}", flush=True)
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    copied_files = 0
    copied_bytes = 0
    last_bytes = 0
    last_log = time.time()
    start = last_log
    for src in src_dir.rglob("*"):
        rel = src.relative_to(src_dir)
        dst = dst_dir / rel
        if src.is_dir():
            dst.mkdir(parents=True, exist_ok=True)
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied_files += 1
        try:
            copied_bytes += src.stat().st_size
        except OSError:
            pass
        now = time.time()
        if copied_bytes - last_bytes >= log_every_bytes or now - last_log >= log_every_seconds:
            pct = 100.0 * copied_bytes / max(1, bytes_total)
            elapsed = max(1e-6, now - start)
            print(
                f"{label}: {pct:5.1f}% {copied_files:,}/{files_total:,} files "
                f"{format_bytes(copied_bytes)}/{format_bytes(bytes_total)} "
                f"[{short_duration(elapsed)}, {format_rate(copied_bytes, elapsed)}]",
                flush=True,
            )
            last_bytes = copied_bytes
            last_log = now
    elapsed = max(1e-6, time.time() - start)
    print(
        f"{label}: copy done {copied_files:,}/{files_total:,} files "
        f"{format_bytes(copied_bytes)} in {short_duration(elapsed)} ({format_rate(copied_bytes, elapsed)})",
        flush=True,
    )
    return dst_dir

# =============================================================================
# USER CONFIG - EDIT ONLY THIS BLOCK
# =============================================================================
RUN_NAME = "parts1_2"

# Full path to the RUKOPYS test folder. This folder must contain metadata.jsonl.
TEST_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset/test")

# Put your YOLO .pt file path here, or put a folder that contains the .pt file.
# Example: YOLO_WEIGHTS_PATH = Path("/kaggle/input/my-yolo-weights/best.pt")
YOLO_WEIGHTS_PATH = Path("")
YOLO_WEIGHT_CANDIDATES = [
    Path("/kaggle/input/models/notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/2/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/htd-box-doclayoutyolo-v4/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/doclayoutyolov4-1/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/input/doclayout-yolo-v4-1/DoclayoutYoloV4.1.pt"),
    Path("/kaggle/working/DoclayoutYoloV4.1.pt"),
]

# Full path to the folder that contains all Qwen weights.
# If base and LoRA are in different Kaggle datasets, set QWEN_BASE_MODEL_DIR and QWEN_LORA_DIR to absolute paths directly.
WEIGHTS_DIR = Path("")
QWEN_BASE_MODEL_DIR = WEIGHTS_DIR / "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1"
QWEN_LORA_DIR = WEIGHTS_DIR / "/kaggle/input/datasets/nguynthikhang/htd-oof-m1/M1"

# Kaggle T4x2 controls.
REQUESTED_NUM_GPUS = 2
REQUIRE_REQUESTED_GPUS = False
TEST_MODE = False  # True = run only first 4 images for a smoke test


# YOLO detection settings.
YOLO_IMG_SIZE = 1280
YOLO_CONF = 0.20
YOLO_MAX_DET = 220
YOLO_IOU_NMS = 0.60
YOLO_DEDUP_IOU = 0.90
YOLO_PAD_SCALE_X = 0.00
YOLO_PAD_SCALE_Y = 0.00

# OCR speed defaults for T4 16GB. OOM is handled by automatically halving the batch.
CROP_BATCH_SIZE = 2
MAX_PIXELS_CROP = 262_144
MAX_NEW_TOKENS_CROP = 192
CROP_PAD_RATIO = 0.04

# Output. The notebook writes only OUTPUT_CSV.
OUTPUT_ROOT_DIR = Path("/kaggle/working/")

# =============================================================================
# INTERNAL CONFIG - USUALLY DO NOT EDIT BELOW
# =============================================================================
KAGGLE_WORKING_DIR = Path("/kaggle/working")
WORK_DIR = KAGGLE_WORKING_DIR / "rukopys_yolo_hybrid_submit"
PARTIAL_DIR = WORK_DIR / "partials"
LOCAL_LORA_CACHE_DIR = WORK_DIR / "lora_adapters"
OUTPUT_DIR = OUTPUT_ROOT_DIR / RUN_NAME
for path in [WORK_DIR, PARTIAL_DIR, LOCAL_LORA_CACHE_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = str(OUTPUT_DIR / f"yolo_hybrid_{RUN_NAME}.csv")
HYBRID_PARTIAL_PREFIX = str(PARTIAL_DIR / f"{RUN_NAME}_yolo_partial_results_gpu")

# Keep the source notebook behavior: OCR every text crop.
CROP_OCR_MODE = "all_text"  # choices: "none", "smart", "all_text"
SORT_CROPS_BY_AREA = True
CLEAR_CUDA_CACHE_EVERY_BATCH = False
COPY_LORA_TO_LOCAL = True
LOAD_LORA_CROP_PROMPTS = False
RESUME_PARTIALS = True
CHECKPOINT_EVERY = 10
PROGRESS_LOG_EVERY = 1
PROGRESS_BAR_WIDTH = 20
USE_TQDM_PROGRESS = False

VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
TEXT_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp", ".bmp"]

YOLO_TYPE_ALIASES = {
    "text": "printed",
    "plain text": "printed",
    "title": "printed",
    "section header": "printed",
    "section-header": "printed",
    "page header": "printed",
    "page-header": "printed",
    "page footer": "printed",
    "page-footer": "printed",
    "caption": "annotation",
    "footnote": "annotation",
    "list item": "printed",
    "list-item": "printed",
    "picture": "image",
    "figure": "image",
    "chart": "graph",
}

SPECIAL_TEXT_MARKER_RULES = (
    "Use these special markers only when they are visible in the crop: "
    "~~word~~ for strikethrough text, ~~old~~{new} for strikethrough text with a visible correction, "
    "and [illegible] for an unreadable word inside an otherwise legible line. "
)

DOCUMENT_CONTEXT_RULES = (
    "The crop may come from one of the following document sources:\n"
    "- Ukrainian dictation handwriting. Do not complete from canonical text; read only visible characters.\n"
    "- Historical Ukrainian/Cyrillic archives and manuscripts. Preserve original spelling, punctuation, and orthography; do not modernize or normalize.\n"
    "- School homework. It may contain corrections, teacher marks, formulas, diagrams, mixed handwriting and printed text.\n"
    "- University exams, coursework, lecture notes, scientific documents, tables, formulas, chemistry notation, mathematics, and technical symbols.\n"
    "General OCR rules:\n"
    "- Read only what is visually present in the image.\n"
    "- Do not infer, reconstruct, autocomplete, or guess missing text.\n"
    "- Do not use memorized canonical versions of poems, dictations, historical texts, exercises, or formulas.\n"
    "- Preserve original spelling, capitalization, punctuation, line breaks, and formatting whenever possible.\n"
    "- Keep uncertain characters exactly as seen; do not silently correct them.\n"
    "- Preserve crossed-out text, corrections, annotations, and teacher marks when visible.\n"
    "- Output only the transcription of visible content."
)

COMMON_OCR_RULES = (
    "Return only the transcription. No JSON, no Markdown, no explanation. "
    + DOCUMENT_CONTEXT_RULES
    + SPECIAL_TEXT_MARKER_RULES
    + "Preserve punctuation, line content, corrections, spelling mistakes, capitalization, digits, "
    "abbreviations, quotes, hyphens, line-final dashes, and visible spacing as much as possible. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, "
    "or infer hidden/missing text."
)

CROP_PROMPTS = {
    "handwritten": (
        "Transcribe the visible handwritten text exactly. "
        "Preserve punctuation, line content, corrections, and strikethrough markers. "
        + COMMON_OCR_RULES
    ),
    "printed": (
        "Transcribe the visible printed or typed text exactly. "
        "Preserve punctuation, line content, corrections, and strikethrough markers. "
        + COMMON_OCR_RULES
    ),
    "annotation": (
        "Read this short annotation, teacher mark, grade, correction, or numbering. "
        "Return only the exact visible text. "
        + SPECIAL_TEXT_MARKER_RULES
        + "No explanation."
    ),
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Do not wrap the "
        "answer in dollar signs. Preserve visible symbols, indices, superscripts, subscripts, arrows, fractions, "
        "matrix/determinant structure, punctuation, numbering, and strikethrough/correction markers. "
        "Do not solve, simplify, normalize, explain, or convert old notation into a different style."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, visible spelling "
        "mistakes, corrections, and strikethrough markers. Do not infer missing cells, rebalance columns, summarize, "
        "or explain."
    ),
    "image": "Return an empty string.",
    "graph": "Return an empty string.",
    "default": "Transcribe the visible content exactly. " + COMMON_OCR_RULES,
}


In [ ]:
def validate_base_model_dir(path):
    path = Path(path)
    if not (path / "config.json").exists():
        raise FileNotFoundError(f"QWEN_BASE_MODEL_DIR must point to the base model folder with config.json: {path}")
    return str(path)


def copy_lora_to_local(src_dir):
    src_dir = Path(src_dir)
    if not (src_dir / "adapter_config.json").exists():
        raise FileNotFoundError(f"QWEN_LORA_DIR must point to the LoRA folder with adapter_config.json: {src_dir}")
    if not COPY_LORA_TO_LOCAL:
        return src_dir

    dst_dir = Path(LOCAL_LORA_CACHE_DIR) / RUN_NAME
    if (dst_dir / "adapter_config.json").exists():
        log_path_stats("Using local LoRA cache", dst_dir)
        return dst_dir
    log_path_stats("LoRA source", src_dir)
    copytree_with_progress(src_dir, dst_dir, "LoRA adapter")
    return dst_dir


def get_test_dir():
    test_dir = Path(TEST_DIR)
    metadata_path = test_dir / "metadata.jsonl"
    if not metadata_path.exists():
        raise FileNotFoundError(f"TEST_DIR must point directly to the test folder with metadata.jsonl: {test_dir}")
    return test_dir


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def normalize_candidate_path(item):
    if item is None:
        return None
    text = str(item).strip()
    if not text or text == ".":
        return None
    return Path(text)


def find_pt_file(path):
    path = Path(path)
    if path.is_file() and path.suffix.lower() in {".pt", ".pth"}:
        return path
    if not path.is_dir():
        return None
    direct = sorted(p for p in path.glob("*.pt") if p.is_file())
    if direct:
        return direct[0]
    nested = sorted(p for p in path.rglob("*.pt") if p.is_file())
    if nested:
        return nested[0]
    return None


def find_yolo_weights():
    checked = []
    for item in [YOLO_WEIGHTS_PATH, *YOLO_WEIGHT_CANDIDATES]:
        path = normalize_candidate_path(item)
        if path is None:
            continue
        checked.append(str(path))
        found = find_pt_file(path)
        if found is not None:
            return found

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for path in sorted(input_root.rglob("*.pt")):
            lower = str(path).lower()
            if "doclayout" in lower or "yolo" in lower:
                return path

    hint = "\n".join(checked[:10]) if checked else "<no explicit candidates>"
    raise FileNotFoundError(
        "No YOLO .pt weights found. Set YOLO_WEIGHTS_PATH to your YOLO weight file or a folder containing it. "
        f"Checked:\n{hint}"
    )


def resolve_image_path(test_dir, file_name):
    test_dir = Path(test_dir)
    raw = Path(file_name)
    name = raw.name
    stem = raw.stem
    candidate_names = [name] + [stem + ext for ext in IMAGE_EXTENSIONS if stem + ext != name]
    candidates = [test_dir / file_name, test_dir / raw.name]
    for candidate_name in candidate_names:
        candidates.extend([test_dir / "images" / candidate_name, test_dir / candidate_name])
    for pp in candidates:
        if pp.exists():
            return str(pp)
    return str(test_dir / "images" / candidate_names[0])


def load_prompt_config(lora_dir):
    global CROP_PROMPTS, MAX_PIXELS_CROP
    cfg_path = Path(lora_dir) / "rukopys_prompt_config.json"
    if not cfg_path.exists():
        return
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if LOAD_LORA_CROP_PROMPTS:
        CROP_PROMPTS.update(cfg.get("crop_prompts", {}))
    MAX_PIXELS_CROP = int(cfg.get("max_pixels_crop", MAX_PIXELS_CROP))


model_id = validate_base_model_dir(QWEN_BASE_MODEL_DIR)
lora_dir = copy_lora_to_local(QWEN_LORA_DIR)
yolo_weights = find_yolo_weights()
test_dir = get_test_dir()
load_prompt_config(lora_dir)

test_records = read_jsonl(test_dir / "metadata.jsonl")
if TEST_MODE:
    test_records = test_records[:4]

log_path_stats("Base model", model_id)
log_path_stats("LoRA", lora_dir)
log_path_stats("YOLO weights", yolo_weights)
print("Run name:", RUN_NAME)
print("Test dir:", test_dir)
print("YOLO weights:", yolo_weights)
print("Images:", len(test_records))
print("Output CSV:", OUTPUT_CSV)


In [ ]:
partial_stem = Path(HYBRID_PARTIAL_PREFIX).name
print("Current hybrid checkpoint files:")
for p in sorted(PARTIAL_DIR.glob(f"{partial_stem}*.csv")):
    try:
        rows = len(pd.read_csv(p).drop_duplicates(subset=["image"], keep="last"))
    except Exception:
        rows = "?"
    print(f"{p} size={p.stat().st_size} rows={rows}")


In [ ]:
def normalize_type(value):
    value = str(value or "handwritten").strip().lower()
    value = value.replace("_", " ").replace("-", " ")
    value = YOLO_TYPE_ALIASES.get(value, value)
    return value if value in VALID_TYPES else "handwritten"


def clamp_xyxy(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    x1, x2 = sorted((max(0, min(width, x1)), max(0, min(width, x2))))
    y1, y2 = sorted((max(0, min(height, y1)), max(0, min(height, y2))))
    if x2 - x1 < 3 or y2 - y1 < 3:
        return None
    return [int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))]


def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    return inter / max(1, area_a + area_b - inter)


def sort_regions(regions):
    return sorted(regions, key=lambda r: (r["bbox"][1], r["bbox"][0]))


def strip_internal_fields(region):
    return {"bbox": region["bbox"], "type": normalize_type(region.get("type")), "text": str(region.get("text") or "")}


def clean_crop_text(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    text = re.sub(r"^(text|transcription|answer)\s*:\s*", "", text, flags=re.I).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
        text = text[1:-1].strip()
    if text.startswith("[") or text.startswith("{"):
        try:
            obj = json.loads(text)
            if isinstance(obj, dict) and "text" in obj:
                text = str(obj["text"])
            else:
                return ""
        except Exception:
            return ""
    return text[:500]


In [ ]:
def get_yolo_names(yolo_model):
    names = getattr(yolo_model, "names", None)
    if names is None and hasattr(yolo_model, "model"):
        names = getattr(yolo_model.model, "names", None)
    return names or {}


def get_class_name(names, cls_id):
    if isinstance(names, dict):
        return names.get(cls_id, names.get(str(cls_id), "handwritten"))
    if isinstance(names, (list, tuple)) and 0 <= cls_id < len(names):
        return names[cls_id]
    return "handwritten"


def load_yolo_model(device):
    print(f"Loading YOLO on {device}: {yolo_weights}", flush=True)
    yolo_model = YOLOv10(str(yolo_weights))
    try:
        yolo_model.to(device)
    except Exception as e:
        print("YOLO .to(device) skipped:", e, flush=True)
    return yolo_model


def dedupe_yolo_regions(regions):
    kept = []
    for region in sorted(regions, key=lambda r: float(r.get("_score", 0.0)), reverse=True):
        duplicate = any(iou(region["bbox"], old["bbox"]) > YOLO_DEDUP_IOU for old in kept)
        if not duplicate:
            kept.append(region)
    return sort_regions([strip_internal_fields(r) for r in kept])


def postprocess_yolo_result(result, img_w, img_h, names):
    boxes = []
    scores = []
    labels = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        boxes.append([x1, y1, x2, y2])
        scores.append(float(box.conf[0]))
        labels.append(int(box.cls[0]))

    if not boxes:
        return []

    boxes_t = torch.tensor(boxes, dtype=torch.float32)
    scores_t = torch.tensor(scores, dtype=torch.float32)
    labels_t = torch.tensor(labels, dtype=torch.int64)

    keep_indices = []
    for cls_id in sorted(set(labels)):
        cls_mask = labels_t == cls_id
        cls_indices = torch.nonzero(cls_mask, as_tuple=True)[0]
        kept = nms(boxes_t[cls_indices], scores_t[cls_indices], YOLO_IOU_NMS)
        keep_indices.extend(cls_indices[kept].tolist())

    regions = []
    for idx in sorted(set(keep_indices)):
        x1, y1, x2, y2 = boxes_t[idx].tolist()
        h = max(1.0, y2 - y1)
        pad_x = YOLO_PAD_SCALE_X * h
        pad_y = YOLO_PAD_SCALE_Y * h
        box = clamp_xyxy([x1 - pad_x, y1 - pad_y, x2 + pad_x, y2 + pad_y], img_w, img_h)
        if box is None:
            continue
        cls_id = int(labels_t[idx].item())
        rtype = normalize_type(get_class_name(names, cls_id))
        regions.append({"bbox": box, "type": rtype, "text": "", "_score": float(scores_t[idx].item())})

    return dedupe_yolo_regions(regions)


def detect_regions_yolo(yolo_model, image_path, yolo_device):
    results = yolo_model.predict(
        source=str(image_path),
        imgsz=YOLO_IMG_SIZE,
        conf=YOLO_CONF,
        max_det=YOLO_MAX_DET,
        verbose=False,
        device=yolo_device,
    )
    result = results[0]
    if hasattr(result, "orig_shape") and result.orig_shape:
        img_h, img_w = result.orig_shape
    else:
        with Image.open(image_path) as img:
            img_w, img_h = img.size
    return postprocess_yolo_result(result, img_w=img_w, img_h=img_h, names=get_yolo_names(yolo_model))


In [ ]:
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig


def configure_processor_for_generation(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"
    return processor


def load_qwen_model(device):
    print(f"Loading base model on {device}: {model_id}", flush=True)
    if str(model_id).startswith("/"):
        log_path_stats("Base model load path", model_id)
    print(f"Loading LoRA adapter: {lora_dir}", flush=True)
    log_path_stats("LoRA load path", lora_dir)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    t0 = time.time()
    base = AutoModelForImageTextToText.from_pretrained(
        model_id,
        device_map={"": device},
        quantization_config=quantization_config,
        dtype=torch.float16,
        trust_remote_code=True,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    print(f"Base model loaded in {format_duration(time.time() - t0)}", flush=True)
    t0 = time.time()
    model = PeftModel.from_pretrained(base, str(lora_dir))
    model.eval()
    print(f"LoRA loaded in {format_duration(time.time() - t0)}", flush=True)
    t0 = time.time()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    processor = configure_processor_for_generation(processor)
    print(f"Processor loaded in {format_duration(time.time() - t0)}", flush=True)
    if processor.tokenizer.pad_token_id is not None:
        model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    return model, processor


def apply_chat_template(processor, messages):
    candidates = [
        {"tokenize": False, "add_generation_prompt": True, "template_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "processor_kwargs": {"enable_thinking": False}},
        {"tokenize": False, "add_generation_prompt": True, "enable_thinking": False},
        {"tokenize": False, "add_generation_prompt": True},
    ]
    for kwargs in candidates:
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message=r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*",
                )
                return processor.apply_chat_template(messages, **kwargs)
        except TypeError:
            continue
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(model, processor, messages_batch, device, max_new_tokens):
    processor.tokenizer.padding_side = "left"
    texts = [apply_chat_template(processor, m) for m in messages_batch]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    try:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            text_kwargs={"padding": True, "return_tensors": "pt"},
            images_kwargs={"return_tensors": "pt"},
            videos_kwargs={"return_tensors": "pt"},
        )
    except TypeError:
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
    inputs = inputs.to(device)
    with torch.inference_mode(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    del inputs, out, trimmed
    if CLEAR_CUDA_CACHE_EVERY_BATCH:
        torch.cuda.empty_cache()
    return decoded


In [ ]:
def resize_to_pixel_budget(img, max_pixels):
    w, h = img.size
    total = max(1, w * h)
    if total <= max_pixels:
        return img
    scale = (max_pixels / total) ** 0.5
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


def crop_image(image_path, bbox):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        w, h = img.size
        x1, y1, x2, y2 = bbox
        pad = int(round(max(x2 - x1, y2 - y1) * CROP_PAD_RATIO))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)
        return resize_to_pixel_budget(img.crop((x1, y1, x2, y2)), MAX_PIXELS_CROP)


def should_crop_ocr(region):
    if CROP_OCR_MODE == "none":
        return False
    if region.get("type") not in TEXT_TYPES:
        return False
    if CROP_OCR_MODE == "all_text":
        return True
    text = str(region.get("text") or "")
    return (not text.strip()) or len(text) < 4 or len(text) > 160


def crop_messages(image_path, region):
    rtype = normalize_type(region.get("type"))
    prompt = CROP_PROMPTS.get(rtype, CROP_PROMPTS["default"])
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": crop_image(image_path, region["bbox"])},
                {"type": "text", "text": prompt},
            ],
        }
    ]


def crop_area_for_sort(region):
    x1, y1, x2, y2 = region.get("bbox", [0, 0, 0, 0])
    return max(1, int(x2 - x1) * int(y2 - y1))


def ocr_regions(qwen_model, processor, image_path, regions, device):
    crop_indices = [i for i, r in enumerate(regions) if should_crop_ocr(r)]
    if SORT_CROPS_BY_AREA:
        crop_indices.sort(key=lambda i: crop_area_for_sort(regions[i]))

    pos = 0
    batch_size = max(1, int(CROP_BATCH_SIZE))
    while pos < len(crop_indices):
        batch_indices = crop_indices[pos:pos + batch_size]
        msgs = [crop_messages(image_path, regions[i]) for i in batch_indices]
        try:
            outs = generate_batch(qwen_model, processor, msgs, device, MAX_NEW_TOKENS_CROP)
        except torch.cuda.OutOfMemoryError as e:
            torch.cuda.empty_cache()
            gc.collect()
            if batch_size > 1:
                new_batch = max(1, batch_size // 2)
                print(f"Crop OCR OOM at batch_size={batch_size}; retrying with batch_size={new_batch}", flush=True)
                batch_size = new_batch
                continue
            print("Crop OCR failed at batch_size=1 with OOM:", e, flush=True)
            pos += 1
            continue
        except Exception as e:
            print("Crop OCR batch failed:", e, flush=True)
            torch.cuda.empty_cache()
            gc.collect()
            if batch_size > 1:
                new_batch = max(1, batch_size // 2)
                print(f"Retrying failed crop batch with batch_size={new_batch}", flush=True)
                batch_size = new_batch
                continue
            pos += 1
            continue

        for idx, out in zip(batch_indices, outs):
            text = clean_crop_text(out)
            if text:
                regions[idx]["text"] = text
        pos += len(batch_indices)
    return sort_regions([strip_internal_fields(r) for r in regions])


def infer_one_image(qwen_model, processor, yolo_model, image_path, device, yolo_device):
    regions = detect_regions_yolo(yolo_model, image_path, yolo_device)
    if not regions:
        return []
    regions = ocr_regions(qwen_model, processor, image_path, regions, device)
    return sort_regions(regions)


In [ ]:
import multiprocessing as mp


def format_duration(seconds):
    if seconds is None or seconds <= 0:
        return "--:--"
    seconds = int(round(seconds))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def progress_bar(done, total, width=PROGRESS_BAR_WIDTH):
    ratio = done / max(1, total)
    filled = min(width, max(0, int(round(width * ratio))))
    return "█" * filled + " " * (width - filled)


def print_progress_line(gpu_id, done, total, run_done, elapsed, region_count, image_name):
    pct = 100.0 * done / max(1, total)
    speed = run_done / elapsed if run_done > 0 and elapsed > 0 else 0.0
    sec_per_img = elapsed / run_done if run_done > 0 else 0.0
    remaining = max(0, total - done)
    eta = remaining / speed if speed > 0 else None
    bar = progress_bar(done, total)
    last = str(image_name or "")[:18]
    if speed > 0:
        timing = f"{format_duration(elapsed)}<{format_duration(eta)}, {sec_per_img:.2f}s/img, speed={speed:.2f} img/s"
    else:
        timing = f"resume, speed=-- img/s"
    print(
        f"GPU {gpu_id}: {pct:3.0f}%|{bar}| {done}/{total} "
        f"[{timing}, regions={region_count}, last={last}]",
        flush=True,
    )


def worker_process(gpu_id, records, output_csv):
    suppress_transformers_noise()
    device = f"cuda:{gpu_id}"
    total_records = len(records)
    print(f"[GPU {gpu_id}] loading Qwen and YOLO for {total_records} images", flush=True)
    qwen_model, processor = load_qwen_model(device)
    yolo_model = load_yolo_model(device)
    print(f"[GPU {gpu_id}] models loaded; starting YOLO + Qwen hybrid inference", flush=True)

    done = set()
    results = []
    if RESUME_PARTIALS and Path(output_csv).exists():
        try:
            old = pd.read_csv(output_csv)
            old = old.drop_duplicates(subset=["image"], keep="last")
            done = set(old["image"].tolist())
            results = old.to_dict("records")
            print(f"[GPU {gpu_id}] resumed {len(done)}/{total_records} rows from {output_csv}", flush=True)
        except Exception as e:
            print(f"[GPU {gpu_id}] could not read checkpoint: {e}", flush=True)

    print_progress_line(gpu_id, len(done), total_records, 0, 0.0, 0, "resumed" if done else "start")

    pbar = None
    if USE_TQDM_PROGRESS:
        pbar = tqdm(
            total=total_records,
            initial=len(done),
            desc=f"GPU {gpu_id}",
            position=gpu_id,
            leave=True,
            dynamic_ncols=True,
            smoothing=0.05,
            unit="img",
            mininterval=1.0,
            maxinterval=10.0,
            file=sys.stdout,
        )
    run_start = time.time()
    run_done = 0

    for rec in records:
        image_name = Path(rec["file_name"]).name
        if image_name in done:
            continue
        image_path = resolve_image_path(test_dir, rec["file_name"])
        try:
            regions = infer_one_image(qwen_model, processor, yolo_model, image_path, device, gpu_id)
        except Exception as e:
            print(f"[GPU {gpu_id}] failed {image_name}: {e}", flush=True)
            regions = []
            torch.cuda.empty_cache()
            gc.collect()
        results.append({"image": image_name, "regions": json.dumps(regions, ensure_ascii=False)})
        done.add(image_name)
        run_done += 1
        elapsed = max(1e-6, time.time() - run_start)
        if pbar is not None:
            pbar.set_postfix_str(
                f"speed={run_done / elapsed:.2f} img/s, last_regions={len(regions)}, last={image_name[:12]}"
            )
            pbar.update(1)
        if run_done % PROGRESS_LOG_EVERY == 0 or len(done) == total_records:
            print_progress_line(gpu_id, len(done), total_records, run_done, elapsed, len(regions), image_name)

        if len(results) % CHECKPOINT_EVERY == 0:
            tmp = output_csv + ".tmp"
            pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
            os.replace(tmp, output_csv)
            if pbar is not None:
                pbar.set_postfix_str(
                    f"speed={run_done / elapsed:.2f} img/s, last_regions={len(regions)}, ckpt={len(results)}"
                )

    if pbar is not None:
        pbar.close()

    tmp = output_csv + ".tmp"
    pd.DataFrame(results).drop_duplicates(subset=["image"], keep="last").to_csv(tmp, index=False)
    os.replace(tmp, output_csv)
    print(f"[GPU {gpu_id}] done {len(done)}/{total_records}; saved {output_csv}", flush=True)


def detect_num_gpus():
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"]).decode("utf-8").strip()
        return max(1, len([x for x in out.splitlines() if x.strip()]))
    except Exception:
        return max(1, torch.cuda.device_count())


def select_num_gpus():
    detected = detect_num_gpus()
    requested = max(1, int(REQUESTED_NUM_GPUS or detected))
    selected = min(detected, requested)
    print(f"GPUs detected={detected}, requested={requested}, using={selected}", flush=True)
    if REQUIRE_REQUESTED_GPUS and detected < requested:
        raise RuntimeError(f"Requested {requested} GPUs but only detected {detected}.")
    if detected < requested:
        print("Warning: fewer GPUs than requested; continuing with the visible GPU count.", flush=True)
    return selected


mp.set_start_method("fork", force=True)
num_gpus = select_num_gpus()

def split_records_for_gpus(records, num_gpus):
    chunk_size = math.ceil(len(records) / num_gpus)
    return [records[i * chunk_size:(i + 1) * chunk_size] for i in range(num_gpus)]


chunks = split_records_for_gpus(test_records, num_gpus)
for gpu_id, chunk in enumerate(chunks):
    print(f"GPU {gpu_id}: assigned {len(chunk)} images", flush=True)

processes = []
partials = []
if num_gpus == 1:
    output_csv = f"{HYBRID_PARTIAL_PREFIX}0.csv"
    partials.append(output_csv)
    print("Single GPU detected; running worker inline for clearer model-load logs.", flush=True)
    worker_process(0, chunks[0], output_csv)
else:
    for gpu_id, chunk in enumerate(chunks):
        if not chunk:
            continue
        output_csv = f"{HYBRID_PARTIAL_PREFIX}{gpu_id}.csv"
        partials.append(output_csv)
        p = mp.Process(target=worker_process, args=(gpu_id, chunk, output_csv))
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    bad_exitcodes = [p.exitcode for p in processes if p.exitcode not in (0, None)]
    if bad_exitcodes:
        raise RuntimeError(f"One or more workers failed with exit codes: {bad_exitcodes}")

frames = []
for path in partials:
    if Path(path).exists():
        frame = pd.read_csv(path)
        print(f"Partial {path}: rows={len(frame)}", flush=True)
        frames.append(frame)
if not frames:
    raise RuntimeError("No partial outputs were created.")

final = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["image"], keep="last")
order = [Path(r["file_name"]).name for r in test_records]
final = final.set_index("image").reindex(order).reset_index()
final["regions"] = final["regions"].fillna("[]")
final.to_csv(OUTPUT_CSV, index=False)
print("Wrote", OUTPUT_CSV, "rows=", len(final), flush=True)
final.head()


In [ ]:
df = pd.read_csv(OUTPUT_CSV)
assert list(df.columns) == ["image", "regions"], df.columns
assert len(df) == len(test_records), (len(df), len(test_records))

bad = []
region_counts = []
for row in df.itertuples(index=False):
    try:
        parsed = json.loads(row.regions)
        assert isinstance(parsed, list)
        region_counts.append(len(parsed))
        for item in parsed:
            assert "bbox" in item and "type" in item and "text" in item
            assert isinstance(item["bbox"], list) and len(item["bbox"]) == 4
            assert item["type"] in VALID_TYPES
    except Exception as e:
        bad.append((row.image, str(e)))
        if len(bad) >= 5:
            break

print("Bad rows:", bad[:5])
print("Images:", len(df))
print("Total regions:", sum(region_counts))
print("Avg regions/page:", round(sum(region_counts) / max(1, len(region_counts)), 2))
print("Ready:", OUTPUT_CSV)
